# Training cost

What PIE and PWPA cost to train on one fold of PIE's benchmark, and how their AUPRC grows with training time and with the number of training knockdowns.
The fold is PIE's `unseen_pert/k562`: training and validation knockdowns of HepG2, Jurkat and RPE1, and those lines' unseen knockdowns as the test set.

- Each model trains on 10%, 25%, 50% and 100% of the fold's training knockdowns. The subsets are nested and hold the same genes in every cell line. Validation and test knockdowns stay the same.
- PIE: PIE's within-dataset recipe as in [`1.train_pie.ipynb`](1.train_pie.ipynb), on one GPU (an L40S, the GPU in the price below).
- PWPA: RFE with early stopping on the validation knockdowns, as in [`0.pie_benchmark.ipynb`](0.pie_benchmark.ipynb), on CPUs only. Its time covers loading the priors and every RFE fit of the three cell lines.
- Each (model, fraction) writes `predictions/cost/<model>/frac_<f>/cost.json` and is skipped when that file exists.

The `MODELS` (`pie,pwpa`) and `FRACTIONS` (`0.1,0.25,0.5,1.0`) environment variables pick the runs.
PIE's runs need PIE's environment (kernel `pie`) and a GPU. PWPA's runs and the figure need this repository's environment with PIE's `src/` on `PYTHONPATH`, for PIE's metric code.
The figure is drawn once all eight files exist: `figures/training_cost.png` and `results/training_cost.tsv`.

In [ ]:
import json
import os
import resource
import sys
import time

import numpy as np
import pandas as pd

sys.path.append("../utils")
from paths import data_dir
from pie_scoring import load, pie_metrics, read_predictions, responses

sys.path.append("../1.model")

ALL_FRACTIONS = [0.1, 0.25, 0.5, 1.0]
MODELS = os.environ.get("MODELS", "pie,pwpa").split(",")
FRACTIONS = [float(f) for f in os.environ.get("FRACTIONS", ",".join(map(str, ALL_FRACTIONS))).split(",")]
FOLD = "k562"
MODELS, FRACTIONS

In [ ]:
pie_dir = data_dir("pie_benchmark")
d = load(pie_dir / "preprocessed")
split = {part: json.loads((pie_dir / "splits" / "unseen_pert" / FOLD / f"{part}.json").read_text()) for part in ("train", "val", "test")}


def rows(part, key):  # key: "replogle.<line>"
    return np.flatnonzero((d.context == key.split(".", 1)[1]) & np.isin(d.pert, split[part][key]))


def auprc(preds):  # (rows, p_de, lfc, delta) per cell line -> PIE's AUPRC, the mean over lines
    return float(pie_metrics(d, *(np.concatenate(x) for x in zip(*preds)))["binary_auprc"])


def cpu_seconds():  # this process and its children
    return sum(getattr(resource.getrusage(who), f) for who in (resource.RUSAGE_SELF, resource.RUSAGE_CHILDREN)
               for f in ("ru_utime", "ru_stime"))


def save(model, f, **record):
    out = data_dir("predictions", "cost", model, f"frac_{f}") / "cost.json"
    out.write_text(json.dumps({"model": model, "fraction": f, "n_train_per_line": n_train[f], **record}, indent=1) + "\n")


# nested subsets: the first k of one permutation of the sorted training genes, the same genes in every cell line
order = np.random.default_rng(0).permutation(np.unique(np.concatenate(list(split["train"].values()))))
subset = {f: set(order[: round(f * len(order))]) for f in ALL_FRACTIONS}
n_train = {f: float(np.mean([len(subset[f] & set(genes)) for genes in split["train"].values()])) for f in ALL_FRACTIONS}
todo = {(m, f) for m in MODELS for f in FRACTIONS if not (data_dir("predictions", "cost", m, f"frac_{f}") / "cost.json").exists()}
n_train, sorted(todo)

PIE trains once per fraction on all three cell lines, and its validation AUPRC is PIE's own, logged at every validation.
A run that is interrupted starts over, so its time is one uninterrupted run.

In [ ]:
for f in sorted(f for m, f in todo if m == "pie"):
    from pie_train import predict, train

    out = data_dir("predictions", "cost", "pie", f"frac_{f}")
    sp = data_dir("predictions", "cost", "pie", f"frac_{f}", "split")
    (sp / "train.json").write_text(json.dumps({k: [g for g in v if g in subset[f]] for k, v in split["train"].items()}, indent=1) + "\n")
    for part in ("val", "test"):
        (sp / f"{part}.json").write_text(json.dumps(split[part], indent=1) + "\n")
    wall, cpu = train(out / "run", pie_dir / "preprocessed", sp, resume=False)
    predict(out / "run", sp / "test.json", out / "pie.parquet")
    curve = pd.read_csv(out / "run" / "validations.csv")
    save("pie", f, wall_seconds=wall, cpu_seconds=cpu, test_auprc=auprc([read_predictions(out / "pie.parquet", d)]),
         curve=curve[["seconds", "val_binary_auprc"]].values.tolist())
    print("pie", f, round(wall), "s", flush=True)

PWPA fits one model per cell line, as in PIE's benchmark.
For the full training set, its curve follows the first fit of each line, on all five priors: at each point the three fits' summed time and their validation AUPRC, each fit held at its last gates once it has stopped.

In [ ]:
for f in sorted(f for m, f in todo if m == "pwpa"):
    os.environ["CUDA_VISIBLE_DEVICES"] = ""  # PWPA trains on CPUs
    import pwpa
    import torch

    torch.set_num_threads(int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count())))

    def partner_average(fit, gates, train, target):
        chosen = [priors[name] for name in fit["priors"]]
        delta, W = pwpa.predict(chosen, fit["scale"], gates, d.pert[target], d.pert[train], responses(d, train))
        p_de, lfc = pwpa.predict_de(W, d.de[train], d.lfc[train], own=d.own[train], tested=d.tested[train].any(0))
        return target, p_de, lfc, delta

    start, cpu = time.perf_counter(), cpu_seconds()
    priors = pwpa.load_priors(data_dir("priors"))  # PIE's five genetic knowledge sources
    fits = {}
    for key in split["test"]:
        train, val, test = rows("train", key), rows("val", key), rows("test", key)
        train = train[np.isin(d.pert[train], list(subset[f]))]
        fit = np.concatenate([train, val])  # validation knockdowns stop the fit early but are never partners
        trace = [] if f == 1.0 else None
        path = pwpa.rfe(priors, [(d.pert[fit], responses(d, fit))], held_out=d.pert[val], trace=trace)
        fits[key] = (train, val, test, pwpa.choose(path), path[0], trace)
    wall, cpu = time.perf_counter() - start, cpu_seconds() - cpu

    test_auprc = auprc([partner_average(best, best["gates"], train, test) for train, _, test, best, _, _ in fits.values()])
    curve = []
    if f == 1.0:  # about 30 log-spaced evaluations
        n = max(len(trace) for *_, trace in fits.values())
        for i in np.unique(np.geomspace(1, n, 30).astype(int)) - 1:
            at = [trace[min(i, len(trace) - 1)] for *_, trace in fits.values()]
            preds = [partner_average(full, gates[None], train, val) for (train, val, _, _, full, _), (_, _, gates) in zip(fits.values(), at)]
            curve.append([sum(t for t, _, _ in at), auprc(preds)])
    save("pwpa", f, wall_seconds=wall, cpu_seconds=cpu, test_auprc=test_auprc, curve=curve)
    print("pwpa", f, round(wall), "s", flush=True)

In [ ]:
import importlib.util

files = {(m, f): data_dir("predictions", "cost", m, f"frac_{f}") / "cost.json" for m in ("pie", "pwpa") for f in ALL_FRACTIONS}
missing = [k for k, p in files.items() if not p.exists()]
if missing:
    print("not drawn yet, missing:", missing)
elif importlib.util.find_spec("matplotlib") is None:
    print("all runs done; draw the figure in this repository's environment (PIE's has no matplotlib)")
else:
    import matplotlib.pyplot as plt

    runs = {k: json.loads(p.read_text()) for k, p in files.items()}

    def context_mean(part):  # each line's mean training knockdown, as 3.score_eight_datasets.ipynb
        preds = []
        for key in split[part]:
            train, target = rows("train", key), rows(part, key)
            m = (d.de[train].mean(0), np.nan_to_num(np.nanmean(d.lfc[train], 0)), d.delta[train].mean(0, dtype=np.float64))
            preds.append((target, *(np.tile(x, (len(target), 1)) for x in m)))
        return auprc(preds)

    b = {"val": context_mean("val"), "test": context_mean("test")}

    # AWS on-demand prices behind the old cost figure (its GEARS bar: 1.4 GPU-hours and 11 CPU core-hours, $3.11 up to rounding):
    # g6e.xlarge (1x L40S) $1.861 per hour; c7i $0.0893 per 2 vCPU-hours, $0.0446 per core-hour
    GPU_HOUR, CORE_HOUR = 1.861, 0.0446
    table = []
    for (model, f), r in runs.items():
        gpu = r["wall_seconds"] / 3600 if model == "pie" else 0.0  # PIE trains on 1 GPU
        core = r["cpu_seconds"] / 3600
        cost = {"n_train_per_line": r["n_train_per_line"], "wall_seconds": r["wall_seconds"], "cpu_seconds": r["cpu_seconds"],
                "gpu_hours": gpu, "cpu_core_hours": core, "aws_usd": gpu * GPU_HOUR + core * CORE_HOUR, "test_auprc": r["test_auprc"]}
        table += [{"model": model, "fraction": f, "quantity": q, "seconds": np.nan, "value": v} for q, v in cost.items()]
        table += [{"model": model, "fraction": f, "quantity": "val_auprc", "seconds": s, "value": v} for s, v in r["curve"]]
    table += [{"model": "context_mean", "fraction": 1.0, "quantity": f"{part}_auprc", "seconds": np.nan, "value": v} for part, v in b.items()]
    table = pd.DataFrame(table)
    table.to_csv("results/training_cost.tsv", sep="\t", index=False)

    PIE, PWPA, GRID, MUTED = "#c87533", "#5a3fa6", "#e3e5ea", "#5f6673"
    plt.rcParams.update({"font.size": 10, "axes.edgecolor": MUTED, "axes.labelcolor": MUTED, "xtick.color": MUTED,
                         "ytick.color": MUTED, "axes.spines.top": False, "axes.spines.right": False})
    fig = plt.figure(figsize=(11, 8.2), layout="constrained")
    top, bottom = fig.subfigures(2, 1, height_ratios=[1, 1.25], hspace=0.06)
    top.suptitle("Training cost on PIE's Replogle–Nadig benchmark", fontweight="bold")
    bottom.suptitle("AUPRC on Replogle–Nadig test knockdowns", fontweight="bold")

    full = table[table.seconds.isna() & (table.fraction == 1.0)].set_index(["quantity", "model"]).value
    COST = {"GPU-hours": ("gpu_hours", "{:.1f}"), "CPU core-hours": ("cpu_core_hours", "{:.1f}"), "AWS on-demand cost": ("aws_usd", "${:.2f}")}
    for ax, (title, (q, fmt)) in zip(top.subplots(1, 3), COST.items()):
        vals = [full[q, "pie"], full[q, "pwpa"]]
        bars = ax.bar(["PIE", "PWPA"], vals, color=[PIE, PWPA], width=0.6)
        ax.bar_label(bars, [fmt.format(v) for v in vals], padding=3, family="monospace", fontsize=9)
        ax.set_title(title, fontsize=10, fontweight="bold")
        ax.set_ylim(0, max(vals) * 1.25)
        ax.yaxis.grid(color=GRID)
        ax.set_axisbelow(True)
        ax.get_xticklabels()[1].set_fontweight("bold")

    left, right = bottom.subplots(1, 2)
    for model, color in (("pie", PIE), ("pwpa", PWPA)):
        curve = np.array(runs[model, 1.0]["curve"])
        left.plot(curve[:, 0] / 60, curve[:, 1], color=color, lw=2)
        n, y = zip(*[(runs[model, f]["n_train_per_line"], runs[model, f]["test_auprc"]) for f in ALL_FRACTIONS])
        right.plot(n, y, color=color, lw=2, marker="o", ms=4)
    left.set_xscale("log")
    left.set_xlabel("Training time (minutes, log scale)")
    left.set_ylabel("Validation AUPRC")
    right.set_xlabel("Training knockdowns per cell line")
    right.set_ylabel("Test AUPRC")
    for ax, part in ((left, "val"), (right, "test")):
        ax.axhline(b[part], color="black", ls=(0, (2, 2)), lw=1)
        ax.text(1.01, b[part], "b", transform=ax.get_yaxis_transform(), va="center", fontweight="bold")
        ax.yaxis.grid(color=GRID)
        ax.set_axisbelow(True)
    handles = [plt.Line2D([], [], color=c, lw=2) for c in (PIE, PWPA)] + [plt.Line2D([], [], color="black", ls=(0, (2, 2)), lw=1)]
    bottom.legend(handles, ["PIE", "PWPA", "b: context mean"], loc="outside lower center", ncol=3, frameon=False)
    fig.savefig("figures/training_cost.png", dpi=200, facecolor="white")